<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Build_a_RAG_System_Dolly_Completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge — Build a Retrieval-Augmented Generation (RAG) System

## Objective

In this notebook, we build a functional RAG system using:

- the `databricks/databricks-dolly-15k` dataset;
- LangChain's `HuggingFaceDatasetLoader`;
- `RecursiveCharacterTextSplitter`;
- `sentence-transformers/all-MiniLM-L6-v2`;
- a FAISS vector store;
- the extractive question-answering model `Intel/dynamic_tinybert`;
- a LangChain `RetrievalQA` chain.

The system retrieves relevant passages and then extracts an answer from the retrieved context. No API key is required.

## 1. Install the required libraries

The additional LangChain packages are included because recent LangChain versions separate community integrations, Hugging Face integrations, text splitters, and legacy chains into dedicated packages.

In [ ]:
%pip install -q -U \
    "datasets<5" \
    "transformers<5" \
    torch \
    sentence-transformers \
    faiss-cpu \
    accelerate \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-classic

In [ ]:
import re
import warnings
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
from pydantic import Field
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    pipeline,
)

from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain_community.vectorstores import FAISS
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE_NUMBER = 0 if torch.cuda.is_available() else -1
DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE_NAME)

## 2. Load the Dolly dataset

`HuggingFaceDatasetLoader` converts dataset rows directly into LangChain `Document` objects.

The exercise requires:

- dataset: `databricks/databricks-dolly-15k`;
- page content column: `context`.

Some Dolly records do not contain reference context. Those empty documents are removed because they cannot contribute useful information to retrieval.

In [ ]:
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(
    path=dataset_name,
    page_content_column=page_content_column,
)

raw_data = loader.load()

print("Documents loaded:", len(raw_data))
print("\nFirst two raw documents:")
for index, document in enumerate(raw_data[:2], start=1):
    print(f"\nDocument {index}")
    print("Content preview:", repr(document.page_content[:500]))
    print("Metadata keys:", list(document.metadata.keys()))

In [ ]:
# Keep only records that provide usable reference context.
data = [
    document
    for document in raw_data
    if document.page_content
    and document.page_content.strip()
]

# Add a stable identifier that will be preserved after chunking.
for document_id, document in enumerate(data):
    document.metadata["document_id"] = document_id
    document.metadata["source"] = dataset_name

print("Non-empty context documents:", len(data))
print("\nExample usable document:")
print(data[0].page_content[:1000])
print("\nExample metadata:")
print(data[0].metadata)

## 3. Split the documents into chunks

The requested parameters are:

- `chunk_size = 1000`;
- `chunk_overlap = 150`.

Overlap helps preserve information that may appear near the boundary between two chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    add_start_index=True,
)

docs = text_splitter.split_documents(data)

chunk_lengths = [len(document.page_content) for document in docs]

print("Original non-empty documents:", len(data))
print("Chunks created:", len(docs))
print("Average chunk length:", round(float(np.mean(chunk_lengths)), 2))
print("Minimum chunk length:", min(chunk_lengths))
print("Maximum chunk length:", max(chunk_lengths))

print("\nFirst chunk:")
print(docs[0].page_content)
print("\nFirst chunk metadata:")
print(docs[0].metadata)

## 4. Generate text embeddings

The model `sentence-transformers/all-MiniLM-L6-v2` converts each chunk into a dense numerical vector.

Similar texts should have vectors that are close to one another, which enables semantic retrieval.

In [ ]:
model_path = "sentence-transformers/all-MiniLM-L6-v2"

model_kwargs = {
    "device": DEVICE_NAME,
}

encode_kwargs = {
    "normalize_embeddings": False,
    "batch_size": 64 if torch.cuda.is_available() else 32,
}

embeddings = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

test_text = "This is a test document."
query_result = embeddings.embed_query(test_text)

print("Embedding model:", model_path)
print("Embedding dimension:", len(query_result))
print("First three values:", query_result[:3])

## 5. Create the FAISS vector store

FAISS indexes the chunk embeddings and allows fast similarity searches.

The whole set of non-empty Dolly context chunks is indexed.

In [ ]:
db = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

print("FAISS vector store created successfully.")
print("Indexed chunks:", len(docs))

## 6. Test retrieval before adding the language model

A retrieval sanity check is essential. It confirms that the vector store returns passages related to the question before the answer-generation component is introduced.

In [ ]:
question = "What is cheesemaking?"

retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retrieved_documents = retriever.invoke(question)

print("Question:", question)
print("Retrieved chunks:", len(retrieved_documents))

for rank, document in enumerate(retrieved_documents, start=1):
    print("\n" + "=" * 100)
    print(f"RANK {rank}")
    print("Document ID:", document.metadata.get("document_id"))
    print("Category:", document.metadata.get("category", "unknown"))
    print("Instruction:", document.metadata.get("instruction", "unknown"))
    print("Start index:", document.metadata.get("start_index", "unknown"))
    print("\nChunk:")
    print(document.page_content[:1500])

## 7. Prepare the Hugging Face question-answering model

`Intel/dynamic_tinybert` is an extractive QA model. It receives:

- a question;
- a context passage;

and returns the most likely answer span found inside that context.

A standard text-generation wrapper cannot directly use this pipeline because extractive QA returns a dictionary rather than newly generated text. The custom `ExtractiveQALLM` class below adapts the QA pipeline to LangChain's LLM interface.

In [ ]:
qa_model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name,
    use_fast=True,
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)

huggingface_qa_pipeline = pipeline(
    task="question-answering",
    model=qa_model,
    tokenizer=tokenizer,
    device=DEVICE_NUMBER,
)

print("Question-answering model loaded:", qa_model_name)

In [ ]:
class ExtractiveQALLM(LLM):
    """
    Adapter that allows an extractive Hugging Face QA pipeline
    to be used inside LangChain RetrievalQA.
    """

    qa_pipeline: Any = Field(exclude=True)
    model_name: str = "Intel/dynamic_tinybert"
    minimum_score: float = 0.01

    @property
    def _llm_type(self) -> str:
        return "huggingface_extractive_question_answering"

    @property
    def _identifying_params(self) -> dict[str, Any]:
        return {
            "model_name": self.model_name,
            "minimum_score": self.minimum_score,
        }

    @staticmethod
    def _extract_question_and_context(prompt: str) -> tuple[str, str]:
        """
        Read the structured prompt created for RetrievalQA.
        """
        pattern = re.compile(
            r"CONTEXT:\s*(.*?)\s*QUESTION:\s*(.*?)\s*ANSWER:\s*$",
            flags=re.IGNORECASE | re.DOTALL,
        )

        match = pattern.search(prompt)

        if not match:
            raise ValueError(
                "The prompt format is invalid. "
                "Expected CONTEXT, QUESTION, and ANSWER sections."
            )

        context = match.group(1).strip()
        question = match.group(2).strip()

        return question, context

    def _call(
        self,
        prompt: str,
        stop: Optional[list[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        question, context = self._extract_question_and_context(prompt)

        if not context:
            return "I could not find relevant context in the dataset."

        prediction = self.qa_pipeline(
            question=question,
            context=context,
            max_seq_len=512,
            doc_stride=128,
            max_answer_len=100,
            handle_impossible_answer=True,
        )

        answer = str(prediction.get("answer", "")).strip()
        score = float(prediction.get("score", 0.0))

        if not answer or score < self.minimum_score:
            return (
                "I could not find a sufficiently supported answer "
                "in the retrieved context."
            )

        return answer


llm = ExtractiveQALLM(
    qa_pipeline=huggingface_qa_pipeline,
    model_name=qa_model_name,
    minimum_score=0.01,
)

print("Custom LangChain extractive QA wrapper created.")

## 8. Build the RetrievalQA chain

The `stuff` strategy joins the retrieved chunks into one context and sends that context to the QA model.

The prompt explicitly separates the context and the question so the custom wrapper can process them reliably.

In [ ]:
prompt_template = '''
Use only the retrieved context to answer the question.
Extract a short answer supported by the context.
If the context does not contain a supported answer, say that no supported answer was found.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
'''.strip()

qa_prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": qa_prompt,
    },
)

print("RetrievalQA chain created successfully.")

## 9. Test the complete RAG system

The result includes:

- the extracted answer;
- the four source chunks selected by FAISS;
- metadata that helps verify where the evidence came from.

In [ ]:
question = "What is cheesemaking?"

result = qa.invoke({
    "query": question,
})

print("=" * 100)
print("QUESTION")
print(question)

print("\nANSWER")
print(result["result"])

print("\nSOURCE DOCUMENTS")
for index, document in enumerate(
    result["source_documents"],
    start=1,
):
    print("\n" + "-" * 100)
    print(f"Source {index}")
    print("Document ID:", document.metadata.get("document_id"))
    print("Category:", document.metadata.get("category", "unknown"))
    print("Instruction:", document.metadata.get("instruction", "unknown"))
    print("Start index:", document.metadata.get("start_index", "unknown"))
    print("Content preview:")
    print(document.page_content[:1000])

## 10. Reusable question-answering function

The helper below makes it easy to test additional questions while preserving the answer and source inspection steps.

In [ ]:
def ask_rag(question: str, show_sources: bool = True) -> dict:
    if not isinstance(question, str) or not question.strip():
        raise ValueError("The question must be a non-empty string.")

    response = qa.invoke({
        "query": question.strip(),
    })

    print("=" * 100)
    print("Question:", question)
    print("\nAnswer:", response["result"])

    source_documents = response.get("source_documents", [])
    print("\nNumber of retrieved sources:", len(source_documents))

    if show_sources:
        for index, document in enumerate(source_documents, start=1):
            print("\n" + "-" * 100)
            print(f"Source {index}")
            print(
                "Instruction:",
                document.metadata.get("instruction", "unknown"),
            )
            print(
                "Category:",
                document.metadata.get("category", "unknown"),
            )
            print(document.page_content[:800])

    return response

In [ ]:
additional_questions = [
    "What ingredients are used to make cheese?",
    "Why is rennet added during cheesemaking?",
    "What is the cheesemaker trying to control?",
]

additional_results = {}

for current_question in additional_questions:
    additional_results[current_question] = ask_rag(
        current_question,
        show_sources=True,
    )
    print("\n")

## 11. Final explanation

### How the RAG system works

1. **Loading:** Dolly's `context` field is loaded as LangChain documents.
2. **Filtering:** Empty contexts are removed.
3. **Chunking:** Long contexts are divided into overlapping chunks.
4. **Embedding:** MiniLM converts each chunk into a semantic vector.
5. **Indexing:** FAISS stores the vectors.
6. **Retrieval:** The question is embedded and compared with the stored vectors.
7. **Question answering:** TinyBERT extracts an answer from the retrieved context.
8. **Verification:** The source chunks are displayed with the answer.

### Why the retriever is important

The QA model cannot search the complete dataset by itself. The retriever narrows thousands of chunks down to the most relevant passages.

### Why documents are chunked

Chunking keeps each passage manageable for the embedding and QA models. The overlap reduces the chance that an important sentence will be separated from its surrounding explanation.

### Role of FAISS

FAISS performs efficient similarity search over the embeddings. It retrieves passages by semantic meaning rather than only by exact word matching.

### Limitation of this implementation

`Intel/dynamic_tinybert` is extractive, so its answer must be a span found in the retrieved text. It does not freely compose a long new explanation like a generative language model. This makes its answers easier to trace to the evidence, but sometimes less natural.

## Optional negative test

A question unrelated to the dataset should produce a weak or unsupported answer. Always inspect the retrieved sources before trusting the result.

In [ ]:
negative_test = ask_rag(
    "What was the final score of yesterday's football match?",
    show_sources=True,
)